# Esperimenti tesi

Il primo esperimento che voglio fare è fittare una serie di alberi con diverse profondità su dei dati sintetici $y \in \{1,-1\}$ con loss $\ell$ e verificare se la differenza tra l'errore su un training set ($\hat R$) e l'errore su un test set ($R$) segue l'andamento previsto dal bound teorico
\begin{equation}
0 \le \mathbb{E}_D[R(\hat f_D)-\hat R(\hat f_D)] \le {2 M} \sqrt{\frac{\log \vert \mathcal{H}\vert}{2n}}
\end{equation}
dove $\ell(\cdot) \le M$, cioè aumenta all'aumentare della profondità del mio albero in un modo controllato. 

N.B. parlo di profondità perché nel caso degli alberi, la cardinalità dell'insieme di ipotesi $\mathcal{H}$ è proporzionale alla profondità, in altre parole più profondo l'albero più comlpesso il modello che impariamo. 

### dati

Costruiamo un dataset sintetico di size $n$, secondo la legge 
\begin{gather}
x \sim \mathrm{Unif}(\text{supp}(0,10))\tag{2}\\
y \coloneqq \mathrm{sign}(f(x) + \varepsilon), \quad \varepsilon \sim \mathcal{N}(0,\sigma^2)\tag{3}
\end{gather}
for some $f: X \to Y$ convex and where $\text{supp}(0,10)$ is a discretization of the interval $[0,10]$ into $N$ points.  

we want 
\begin{equation}
n_\text{train} \ll N \ll n_\text{test} \approx \infty
\end{equation}

### modelli e plot
we train **binary trees** with depths ranging from 1 to a multiple of the $\log_2 n$ threshold to see long term trends, using a **counting loss**. 

we compute the expected values of $R$ and $\hat R$ (and their difference) over the disorder of the data $D$ averaging over a number of training samples equal to ```num_iter```. 

we then do a number of plots to visualize to what extent empirical results confirm the theoretical bound $\text{(1)}$. 


In [2]:
print('mare')

mare


In [ ]:
# build dataset

import numpy as np

rng = np.random.default_rng(seed=4)  

def sample_x(data_size, supp_size, l=0.0, u=10.0, rng=rng):
    """sample n points from supp evenly spaced points on [l, u]"""
    support = np.linspace(l, u, supp_size)
    return rng.choice(support, size=data_size, replace=True)

def sample_y(x, f, sigma=1.0, rng=rng):
    """y = sign(f(x) + e), e ~ N(0, sigma)"""
    e = sigma * rng.standard_normal(size=len(x))
    return np.sign(f(x) + e)

def sample_y_bernoulli(x, f, p=0.8, rng=rng):
    """y = sign(f(x)) * b, b = +1 w.p. p, b = -1 w.p. 1-p"""
    b = rng.choice(np.array([1, -1]), size=len(x), p=[p, 1 - p])
    return np.sign(f(x)) * b

# explore dataset

def explore_data(fun, n_train, supp_size, l=0.0, u=10.0, sigmas=(10**(-4), 1, 5, 10),
                  n_grid=20_000, error_type='gaussian', probs=(0.5, 0.7, 0.9, 0.99)):
    '''
    plot the data and a sample training set to see where the points are 
    and how the noise influences the distribution.

    error_type selects which noise model to sweep over the 4 panels:
    'gaussian' sweeps sigmas (points scattered at f(x)+e, so the noise's
    effect on the position is visible); 'bernoulli' sweeps probs (there's no
    continuous noisy value here, just a sign flip, so points are scattered
    on the f(x) curve itself and only their color shows the flip).

    points are colored by whether the sampled y agrees with the noiseless
    sign(f(x)) — blue for correctly classified, red for misclassified — and
    each panel title also reports the misclassified count, a direct read of
    how much the noise level actually flips labels.
    '''
    x_plot = np.linspace(l, u, n_grid)

    # same x locations across panels so only the noise differs
    x = sample_x(data_size=n_train, supp_size=supp_size, l=l, u=u)
    true_y = np.sign(fun(x))

    # x-values where fun crosses the decision boundary (fun(x) = 0), found by
    # linear interpolation between sign changes of fun on the dense grid x_plot
    f_plot = fun(x_plot)
    crossing_idx = np.where(np.diff(np.sign(f_plot)) != 0)[0]
    x0, x1 = x_plot[crossing_idx], x_plot[crossing_idx + 1]
    y0, y1 = f_plot[crossing_idx], f_plot[crossing_idx + 1]
    roots = x0 - y0 * (x1 - x0) / (y1 - y0)

    fig, axs = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
    if error_type == 'gaussian':
        fig.suptitle(r'data distribution for different noise levels $\sigma$', fontsize=13)
        params = sigmas
    elif error_type == 'bernoulli':
        fig.suptitle(r'data distribution for different flip probabilities $p$', fontsize=13)
        params = probs

    for ax, param in zip(axs.flat, params):
        if error_type == 'gaussian':
            e = param * rng.standard_normal(size=len(x))
            noisy_f = fun(x) + e
            y = np.sign(noisy_f)
            panel_title = rf'$\sigma$ = {param}'
        elif error_type == 'bernoulli':
            y = sample_y_bernoulli(x, fun, p=param)
            noisy_f = fun(x)
            panel_title = rf'$p$ = {param}'

        misclassified = y != true_y
        n_misclassified = np.sum(misclassified)
        panel_title += f'  ({n_misclassified}/{n_train} misclassified)'

        ax.plot(x_plot, fun(x_plot), color='#0b0b0b', linewidth=1, label=r'$f(x)$')
        ax.axhline(0, color='#c3c2b7', linestyle='--', linewidth=1.2, label='decision boundary')
        for i, root in enumerate(roots):
            ax.axvline(root, color='#898781', linestyle=':', linewidth=1.5,
                       label=r'$f(x)=0$' if i == 0 else None)
        ax.scatter(x[~misclassified], noisy_f[~misclassified], color='#2a78d6', s=20, alpha=0.8, label='correctly classified')
        ax.scatter(x[misclassified],  noisy_f[misclassified],  color='#e34948', s=20, alpha=0.8, label='misclassified')

        ax.set_title(panel_title)
        ax.set_xlabel('x')
        ax.set_ylabel(r'$f(x)+\varepsilon$' if error_type == 'gaussian' else r'$f(x)$')
        ax.grid(True, alpha=0.3)

    axs.flat[0].legend(loc='upper center')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# almost-always-fixed details

def get_model_class(model):
    """works for both a class (trees) and a functools.partial (e.g. MLP)"""
    return model.func if hasattr(model, 'func') else model

def get_depths_list(t, model):
    model_class = get_model_class(model)
    if model_class == DecisionTreeClassifier:
        depths = np.arange(1, int(np.floor(7 * t)) + 1)
        depths_float = depths.astype(float)
    elif model_class == MLPClassifier:
        max_pow = int(np.ceil(np.log2(t * 7)))
        depths = np.array([2**i for i in range(max_pow + 1)], dtype=float)
        depths_float = depths
    elif model_class == RandomForestClassifier:
        # hybrid complexity measure (Belkin et al.'s random forest double-
        # descent experiment): first grow a single tree (n_estimators=1)
        # deeper and deeper until it's complex enough to interpolate, then
        # freeze the tree fully grown (max_depth=None) and instead grow the
        # ensemble by adding more trees. there's no overfitting-threshold
        # scale for this hybrid measure, so t is unused and these ranges are
        # fixed rather than derived from n_train.
        max_single_tree_depth = 20
        max_n_estimators = 200

        depth_phase        = np.arange(1, max_single_tree_depth + 1)
        n_estimators_phase = np.arange(2, max_n_estimators + 1)

        # column 0: max_depth to use (-1 is a sentinel for "None", i.e. fully
        # grown/interpolating); column 1: n_estimators to use
        phase_one = np.column_stack([depth_phase, np.ones_like(depth_phase)])
        phase_two = np.column_stack([-np.ones_like(n_estimators_phase), n_estimators_phase])
        depths = np.concatenate([phase_one, phase_two], axis=0)

        # depth and n_estimators live on different scales, so a plain
        # increasing index is used as the x-axis / complexity value instead
        depths_float = np.arange(1, len(depths) + 1, dtype=float)
    return depths, depths_float

def centered_parabola(x):
    return (x-5)**2 - 6.25

# define some sine functions with fixed height at 1.2 and given number of crossings in the interval [0,10 ]
def sine(x, num_crossings, u = 10.0):
    # obtain a number of crossings equal to the given one, in the interval [0, u]
    c = (np.pi * (num_crossings +1)) / u
    return 5*np.sin(c*x)

# build model and create plots

from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def main_loop(x_test, y_test, n_train, sigma, num_iter, model,  N=10_000,
              l = 0.0, u = 10.0, fun=centered_parabola, error_type = 'gaussian', error_par = 0.8):
    
    '''
    function to build models and obtain the exp values of R, R_hat, and their difference for a number of depths.

    sample num_iter training sets once, then for each depth:
    fit model to each of those same num_iter training sets;
    compute the metrics;
    average at the end to cancel the disorder over the data randomness.

    reusing the same num_iter datasets at every depth (instead of drawing
    num_iter fresh ones per depth) means depth is the only thing that changes
    across the curve — the disorder D is common across depths, so depth-to-
    depth differences aren't also mixed with independent sampling noise.

    also keeps, for each depth, the fitted model (among the num_iter) that
    minimizes the training error hat_R, so it can be plotted/animated later
    as "the" learned function for that depth.

    return lists of the 3 metrics, one value for each depth, plus the best
    prediction/sample per depth and the x_grid they were evaluated on.
    '''

    # used for plotting best prediction
    x_grid = np.linspace(l, u, N)

    # overfitting threshold: left as None for models with no threshold logic
    # defined here (e.g. RandomForestClassifier's hybrid complexity measure),
    # so the plots simply skip drawing it
    threshold = None
    if get_model_class(model) == DecisionTreeClassifier:
        threshold = np.log2(n_train)
    
    elif get_model_class(model) == MLPClassifier:
        # ~3 params per hidden unit with 1D input → threshold at n_train/3
        threshold = n_train / 3

    depths_list, _ = get_depths_list(threshold, model)

    # sample the num_iter training sets once: reused, unchanged, at every depth
    xsamples = []; ysamples = []
    for i in range(num_iter):
        x = sample_x(n_train, N)
        if error_type == 'gaussian':
            y = sample_y(x, fun, sigma)
        elif error_type == 'bernoulli':
            y = sample_y_bernoulli(x, fun, p = error_par)
        xsamples.append(x); ysamples.append(y)

    E_hat_Rs = []; E_Rs = []; E_gaps = []
    std_hat_Rs = []; std_Rs = []; std_gaps = []
    best_predictions = []; best_xsample = []; best_ysample = []
    for d in depths_list:
        # initialize for each depth the list of R hats, Rs, and gaps (will do mean) 
        hat_Rs = []; Rs = []; gaps = []; 
        # and the quantities for plotting best function (will do argmin)
        predictions = []
        for i in range(num_iter):
            x, y = xsamples[i], ysamples[i]
            # fit model to training data
            if get_model_class(model) == DecisionTreeClassifier:
                clf = model(max_depth=int(d), random_state=42)
            elif get_model_class(model) == MLPClassifier:
                clf = model(hidden_layer_sizes=(int(d),))
            elif get_model_class(model) == RandomForestClassifier:
                # d is a (max_depth_or_sentinel, n_estimators) row from
                # get_depths_list's hybrid measure: -1 means "fully grown"
                depth_val, n_estimators_val = int(d[0]), int(d[1])
                max_depth = None if depth_val == -1 else depth_val
                clf = model(max_depth=max_depth, n_estimators=n_estimators_val, random_state=42)
            clf.fit(x.reshape(-1, 1), y)
            predictions.append(clf.predict(x_grid.reshape(-1, 1)))

            # compute errors and gap
            train_err = 1 - clf.score(x.reshape(-1, 1), y)
            test_err  = 1 - clf.score(x_test.reshape(-1, 1), y_test)
            gap = test_err - train_err
            hat_Rs.append(train_err); Rs.append(test_err); gaps.append(gap)

        # store values for ploting best learned function
        best_idx = 1
        # int(np.argmin(hat_Rs))
        best_predictions.append(predictions[best_idx])
        best_xsample.append(xsamples[best_idx])
        best_ysample.append(ysamples[best_idx])

        # out of the num_iter loop, we append the avg value for the current depth d
        E_hat_Rs.append(np.mean(hat_Rs))
        E_Rs.append(np.mean(Rs))
        E_gaps.append(np.mean(gaps))
        # and the std across the num_iter samples, for the variance bands in plots()
        std_hat_Rs.append(np.std(hat_Rs))
        std_Rs.append(np.std(Rs))
        std_gaps.append(np.std(gaps))

        print(f'Depth: ', d, 'R cappuccios: ', hat_Rs)
    return (np.array(E_hat_Rs), np.array(E_Rs), np.array(E_gaps),
            np.array(std_hat_Rs), np.array(std_Rs), np.array(std_gaps),
            best_predictions, best_xsample, best_ysample, x_grid)


def plots(E_hat_Rs, E_Rs, E_gaps,
          # variables for visualizations in the plots
          n_train, sigma, num_iter,
          # variable for obtaining useful quantities
          model,
          # std across the num_iter samples (as returned by main_loop()), used
          # to draw a shaded band of ± 1 std around the corresponding curve;
          # left as None to skip the band, e.g. when plotting from data that
          # doesn't carry a std (older call sites, or axes reused across runs),
          bound,
          std_hat_Rs=None, std_Rs=None, std_gaps=None,
          one=True, two=True, three=True,
          axes=None, error_type = 'gaussian', error_par = 0.8):

    '''
    function to obtain the following plots, given the metrics obtained with main_loop():

    1) classical train vs. test error plot, against depth of the model;
    2) plots the empirical gap and the theoretical function that should upper bound the gap depending on the depth;
    3) plots the same as (2) with as RHS E[hat_R] + bound.

    each curve coming from main_loop's num_iter samples (E_hat_Rs, E_Rs,
    E_gaps) can be shown with a shaded ± 1 std band behind it, in a lighter
    shade of the same color, if the corresponding std array is passed.
    '''
    
    # depths and quantities for plotting overft threshold and the bound function.
    # threshold is left as None for models with no threshold logic defined
    # here (e.g. RandomForestClassifier's hybrid complexity measure), so the
    # "if threshold:" checks below simply skip drawing it
    threshold = None
    if get_model_class(model) == DecisionTreeClassifier:
        threshold = np.log2(n_train)

    elif get_model_class(model) == MLPClassifier:
        # ~3 params per hidden unit with 1D input → threshold at n_train/3
        threshold = n_train / 3

    _, depths_list_float = get_depths_list(threshold, model)
    # bound = np.sqrt((2* depths_list_float) / (n_train))

    # cluade logic for getting multiple plots at a time
    n_plots = sum([one, two, three])
    if n_plots == 0:
        return
    if axes is None:
        fig, axs = plt.subplots(n_plots, 1, figsize=(10, 5 * n_plots))
        if n_plots == 1:
            axs = [axs]
        if error_type == 'gaussian':
            fig.suptitle(f'Train and test errors against model complexity\n($n_{{\mathrm{{train}}}}$={n_train}, $\sigma$={sigma}, $n_{{\mathrm{{iter}}}}$={num_iter})', fontsize=13, weight = 'normal')
        elif error_type == 'bernoulli':
            fig.suptitle(f'Train and test errors against model complexity\n($n_{{\mathrm{{train}}}}$={n_train}, $p$={error_par}, $n_{{\mathrm{{iter}}}}$={num_iter})', fontsize=13, weight = 'normal')

    else:
        axs = axes
    ax_iter = iter(axs)

    if one:
        #1: grafico solito train e test error
        ax = next(ax_iter)
        zero_train = next((d for d, e in zip(depths_list_float, E_hat_Rs) if e == 0), None)
        ax.plot(depths_list_float, E_hat_Rs, color='#FF007F', marker='o', label=r'$E[\hat R]$')
        ax.plot(depths_list_float, E_Rs,     color='#00CED1', marker='o', label=r'$E[R]$')
        if std_hat_Rs is not None:
            ax.fill_between(depths_list_float, E_hat_Rs - std_hat_Rs, E_hat_Rs + std_hat_Rs,
                             color='#FF007F', alpha=0.15, linewidth=0)
        if std_Rs is not None:
            ax.fill_between(depths_list_float, E_Rs - std_Rs, E_Rs + std_Rs,
                             color='#00CED1', alpha=0.15, linewidth=0)
        # logica per if-else above (overft threshold a 0, se computato esplicitamente, non ha senso quindi ignoro il caso)
        if threshold:
            ax.axvline(threshold, color='red',  linestyle='--', linewidth=1.2, label='theoretical overfitting th.')
        # questo è per plottare empirical overft threshold
        # if zero_train is not None:
        #     ax.axvline(zero_train, color='blue', linestyle='--', linewidth=1.2, label='empirical overfitting th.')
        ax.set_ylim(bottom=0)
        ax.set_title(rf'train vs test error')
        ax.set_xlabel('depth'); ax.set_ylabel('error')
        ax.legend(); ax.grid(True, alpha=0.3)

    if two:
        #2: grafico che confronta il bound teorico e il gap empirico
        ax = next(ax_iter)
        zero_train = next((d for d, e in zip(depths_list_float, E_hat_Rs) if e == 0), None)
        ax.plot(depths_list_float, E_gaps, color='#32CFFFFF', marker='o', label=r'$E[R - \hat R]$')
        ax.plot(depths_list_float, bound,  color='gray', linestyle='--', label=r'$\sqrt{\frac{2d}{n}}$')
        if std_gaps is not None:
            ax.fill_between(depths_list_float, E_gaps - std_gaps, E_gaps + std_gaps,
                             color='#32CFFFFF', alpha=0.15, linewidth=0)
        if threshold:
            ax.axvline(threshold, color='red',  linestyle='--', linewidth=1.2, label='theoretical overfitting th.')
        # if zero_train is not None:
        #     ax.axvline(zero_train, color='blue', linestyle='--', linewidth=1.2, label='empirical overfitting th.')
        ax.set_ylim(bottom=0)
        ax.set_title(rf'gap $E[R-\hat R]$ and theoretical bound')
        ax.set_xlabel('depth'); ax.set_ylabel('gap')
        ax.legend(); ax.grid(True, alpha=0.3)

    if three:
        #3: grafico che confronta la LHS E[R] con la RHS E[hat_R] + bound
        ax = next(ax_iter)
        zero_train = next((d for d, e in zip(depths_list_float, E_hat_Rs) if e == 0), None)
        RHS = E_hat_Rs + bound
        ax.plot(depths_list_float, E_Rs,  color='#FD6428FF', marker='o', label=r'$E[R]$')
        ax.plot(depths_list_float, RHS,   color='#6BC615FF', linestyle='--', marker='s', label=r'$E[\hat R] + 2\sqrt{\frac{\log d}{2n}}$')
        if std_Rs is not None:
            ax.fill_between(depths_list_float, E_Rs - std_Rs, E_Rs + std_Rs,
                             color='#FD6428FF', alpha=0.15, linewidth=0)
        if std_hat_Rs is not None:
            # RHS's randomness comes only from E_hat_Rs (the bound term is deterministic)
            ax.fill_between(depths_list_float, RHS - std_hat_Rs, RHS + std_hat_Rs,
                             color='#6BC615FF', alpha=0.15, linewidth=0)
        if threshold:
            ax.axvline(threshold, color='red',  linestyle='--', linewidth=1.2, label='theoretical overfitting th.')
        # if zero_train is not None:
        #     ax.axvline(zero_train, color='blue', linestyle='--', linewidth=1.2, label='empirical overfitting th.')
        ax.set_ylim(bottom=0)
        ax.set_title(rf'$E[R]$ and $E[\hat{{R}}]$ + bound')
        ax.set_xlabel('depth'); ax.set_ylabel('error')
        ax.legend(); ax.grid(True, alpha=0.3)

    if axes is None:
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()


def animate_learned_function(model, n_train, best_pred, best_x, best_y, x_grid,
                             fun=centered_parabola, interval=600):
    '''
    build the animation of the learned function across depths from
    already-computed per-depth predictions/samples (e.g. the ones returned by
    main_loop(), or by animate_learned_function() below): for each depth,
    shows the training points that produced the plotted fit and the
    corresponding learned function, alongside sign(f(x)).
    '''
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(x_grid, np.sign(fun(x_grid)), color='#0b0b0b', linewidth=1.5, linestyle='--', label=r'sign$(f(x))$')
    scat = ax.scatter([], [], color='#2a78d6', s=30, alpha=0.7, zorder=3, label='training data')
    line, = ax.plot([], [], color='#e34948', linewidth=2, label='learned function')
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel('x'); ax.set_ylabel('predicted class')
    ax.legend(loc='center right')
    ax.grid(True, alpha=0.3)

        # overfitting threshold: left as None for models with no threshold
        # logic defined here (e.g. RandomForestClassifier's hybrid measure)
    threshold = None
    if get_model_class(model) == DecisionTreeClassifier:
        threshold = np.log2(n_train)
    
    elif get_model_class(model) == MLPClassifier:
        # ~3 params per hidden unit with 1D input → threshold at n_train/3
        threshold = n_train / 3

    depths_list, _ = get_depths_list(threshold, model)
    is_forest = get_model_class(model) == RandomForestClassifier

    def update(frame):
        line.set_data(x_grid, best_pred[frame])
        scat.set_offsets(np.column_stack([best_x[frame], best_y[frame]]))
        if is_forest:
            depth_val, n_estimators_val = int(depths_list[frame][0]), int(depths_list[frame][1])
            depth_label = 'full' if depth_val == -1 else depth_val
            ax.set_title(f'depth={depth_label}, n_trees={n_estimators_val}')
        else:
            ax.set_title(f'depth = {int(depths_list[frame])}')
        return line, scat

    anim = FuncAnimation(fig, update, frames=len(depths_list), interval=interval)
    plt.close(fig)
    return anim

def animate_tree_splits(model, n_train, best_pred, best_x, best_y, x_grid,
                         fun=centered_parabola, interval=600):
    '''
    visualize how a decision tree splits the 1D input into branches as depth
    grows: for each depth, shades every leaf's region of x_grid by its
    predicted class, draws a dashed vertical line at each split threshold
    between adjacent leaves, and scatters the training points (colored by
    their true label) at their (x, y) position. as depth increases and more
    lines/leaves appear, the training points visually end up sorted into
    thinner and thinner regions — literally the effect of the tree's splits
    partitioning the 50 datapoints into more and more branches.

    only meaningful for DecisionTreeClassifier, the only model here whose
    predictions come from an explicit axis-aligned partition of the input.

    reuses the predictions/samples already computed by main_loop() (passed
    in as best_pred/best_x/best_y/x_grid) instead of refitting: at depth d,
    best_pred[d] is the predicted class along x_grid, and the points where it
    changes are exactly the tree's split thresholds at that depth.
    '''
    assert get_model_class(model) == DecisionTreeClassifier, \
        'animate_tree_splits only supports DecisionTreeClassifier'

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.set_xlim(x_grid.min(), x_grid.max())
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.grid(True, alpha=0.3)

    threshold = np.log2(n_train)
    depths_list, _ = get_depths_list(threshold, model)

    scat = ax.scatter([], [], s=30, zorder=3, edgecolor='k', linewidth=0.4)
    title = ax.set_title('')
    artists = {'regions': [], 'lines': []}

    def update(frame):
        for p in artists['regions']: p.remove()
        for l in artists['lines']: l.remove()
        artists['regions'] = []; artists['lines'] = []

        pred = best_pred[frame]
        # leaf boundaries: midpoints between consecutive x_grid points where
        # the predicted class changes (i.e. where a split separates leaves)
        change_idx = np.where(np.diff(pred) != 0)[0]
        boundaries = ([x_grid[0]]
                       + [(x_grid[i] + x_grid[i + 1]) / 2 for i in change_idx]
                       + [x_grid[-1]])

        for start, end in zip(boundaries[:-1], boundaries[1:]):
            mid_idx = np.searchsorted(x_grid, (start + end) / 2)
            leaf_class = pred[min(mid_idx, len(pred) - 1)]
            color = '#2a78d6' if leaf_class == 1 else '#e34948'
            artists['regions'].append(ax.axvspan(start, end, color=color, alpha=0.12, zorder=0))

        for b in boundaries[1:-1]:
            artists['lines'].append(ax.axvline(b, color='#555', linestyle='--', linewidth=1, zorder=1))

        x, y = best_x[frame], best_y[frame]
        scat.set_offsets(np.column_stack([x, y]))
        scat.set_color(np.where(y == 1, '#2a78d6', '#e34948'))

        n_leaves = len(boundaries) - 1
        title.set_text(f'depth = {int(depths_list[frame])}, leaves = {n_leaves}')
        return [scat, title, *artists['regions'], *artists['lines']]

    anim = FuncAnimation(fig, update, frames=len(depths_list), interval=interval)
    plt.close(fig)
    return anim


In [ ]:
''' 
define some bounds!
apart from the first explorative tasks, this is the most mathematically insightful part in the code, where all the theory until now is implemented: we define multiple bounds and see which one captures the behavior of the LHS (test error) best

N.B. i bound contengono depth come input qui, che entra in modo lineare, perché la misura di complessità nella teoria è la cardinalità (exp(depth)) che entra in modo logaritmico (avendo io le depths in questo codice, risulta più semplice fare così a costo della consistency con la teoria)
'''

def bound_naive(depth, n_train, M=1):
    # Thm 2.5 or inequality (43) up to constants
    return 2*M*(np.sqrt((2* depth) / (n_train)))

def bound_hans():
    # Prop 2.9 using empirical Rademacher
    emp_rademacher = 7
    return 2*emp_rademacher

def bound_variance(depth, n_train, inf_R):
    # Inequality (47)
    return 

def bound_noise(depth, n_train, inf_R, alpha, h):
    # Inequality (49)
    pass